# 🚢 Supply Chain Agent Workshop
## Building an AI Agent from Scratch with LangChain

### What you will build:
An AI agent that monitors supply chain risks by searching the web in real-time.

### What you will learn:
| Concept | What it means | Why it matters |
|---|---|---|
| **Human-in-the-Loop** | Agent asks a human before acting | Keeps humans in control of high-stakes decisions |
| **Model Flexibility** | Swap GPT ↔ Claude ↔ Gemini easily | You're never stuck with one vendor |
| **Observability** | Every step is logged and traceable | You can debug, audit, and improve your agent |
| **No Vendor Lock-In** | One codebase, any LLM | Business continuity if a provider changes pricing |

---
**Before you start:** Make sure you've copied `.env.example` to `.env` and filled in your API keys.

## Part 1: Setup and Configuration
Run this cell first to install dependencies and verify your config.

In [ ]:
# Install dependencies (first run only)
# !pip install -r ../requirements.txt

import sys, os
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

# Set environment variables for LangSmith tracing
os.environ['LANGCHAIN_TRACING_V2'] = os.getenv('LANGCHAIN_TRACING_V2', 'false')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY', '')
os.environ['LANGCHAIN_PROJECT'] = 'supply-chain-workshop'

print('✅ Environment loaded')
print(f'   Provider: {os.getenv("LLM_PROVIDER", "openai")}')
print(f'   LangSmith: {os.getenv("LANGCHAIN_TRACING_V2", "false")}')

## Part 2: Understanding the Tools
Tools are how the agent interacts with the outside world.
Each tool is a Python function + a description the LLM reads.

> **Key Insight:** The LLM decides WHICH tool to use based on the description.
> Writing good tool descriptions is a critical skill.

In [ ]:
from langchain_community.utilities import GoogleSearchAPIWrapper
from langchain_community.tools import Tool

# Initialize the Google Search wrapper
search = GoogleSearchAPIWrapper()

# Test the search tool directly (before connecting it to the agent)
result = search.run('Port of Vancouver shipping delays 2025')
print('🔍 Direct Search Result:')
print(result[:500])

In [ ]:
# Now wrap it as a LangChain Tool
# The 'description' is what the LLM reads to decide when to use this tool

supply_chain_search = Tool(
    name='supply_chain_search',
    func=search.run,
    description=(
        'Search for current supply chain news, disruptions, port delays, '
        'supplier financial issues, trade route problems, or tariff changes. '
        'Input should be a specific search query.'
    ),
)

# WORKSHOP EXERCISE: Add a second tool for checking freight rates
# Hint: use search.run with a query that includes 'freight rate' or 'shipping cost'
# YOUR CODE HERE:
# freight_rate_tool = Tool(
#     name='...',
#     func=...,
#     description='...',
# )

tools = [supply_chain_search]
print(f'✅ {len(tools)} tool(s) ready')
for t in tools:
    print(f'   - {t.name}: {t.description[:80]}...')

## Part 3: Observability — Watching the Agent Think
Before we build the agent, let's build the logging system.

> **Key Insight:** Without observability, an agent is a black box.
> With it, you can see every thought, every tool call, every result.
> This is essential for debugging and for building client trust.

In [ ]:
from langchain.callbacks.base import BaseCallbackHandler
from langchain.schema import AgentAction, AgentFinish, LLMResult

class WorkshopObservabilityHandler(BaseCallbackHandler):
    """Logs every step the agent takes. Add your own logging here."""
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        print('\n🧠 Agent is thinking...')
    
    def on_tool_start(self, serialized, input_str, **kwargs):
        print(f'\n🔧 Using tool: {serialized["name"]}')
        print(f'   Query: {input_str[:100]}')
    
    def on_tool_end(self, output, **kwargs):
        print(f'📄 Got result ({len(output)} chars)')
    
    def on_agent_finish(self, finish: AgentFinish, **kwargs):
        print('\n✅ Agent finished!')

    # WORKSHOP EXERCISE: Add a method to log token usage
    # Hint: override on_llm_end and inspect response.llm_output

handler = WorkshopObservabilityHandler()
print('✅ Observability handler ready')

## Part 4: Model Flexibility
This is the section that prevents vendor lock-in.
We'll build a factory function that returns any LLM.

> **Key Insight:** The agent code never references OpenAI or Anthropic directly.
> It only works with a generic `BaseChatModel`. This is the abstraction layer.

In [ ]:
def get_llm(provider='openai', model_name=None, temperature=0.0):
    """
    Workshop demo: swap providers by changing ONE argument.
    The agent below never needs to change.
    """
    if provider == 'openai':
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model_name or 'gpt-4o-mini', temperature=temperature)
    
    elif provider == 'anthropic':
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model_name or 'claude-3-haiku-20240307', temperature=temperature)
    
    elif provider == 'google':
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model_name or 'gemini-1.5-flash', temperature=temperature)
    
    raise ValueError(f'Unknown provider: {provider}')

# Test that it works (will fail if API key is missing — that's expected)
try:
    llm = get_llm(provider='openai')
    print(f'✅ LLM ready: {llm}')
except Exception as e:
    print(f'⚠️  LLM init error (check your API key): {e}')

## Part 5: Building the Agent
Now we put it all together.

The **ReAct pattern** (Reason + Act) is the most common agent pattern:
1. **Thought**: What do I need to do?
2. **Action**: Call a tool
3. **Observation**: Read the tool result
4. Repeat until ready to answer

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate

PROMPT = PromptTemplate.from_template("""
You are a Supply Chain Risk Analyst. Search for risks and report findings clearly.

Tools: {tools}
Tool names: {tool_names}

Format:
Thought: [reasoning]
Action: [tool_name]
Action Input: [search query]
Observation: [tool result]
... repeat as needed ...
Thought: I have enough information.
Final Answer: [risk assessment with severity: LOW/MEDIUM/HIGH]

Question: {input}
{agent_scratchpad}
""")

def build_agent(provider='openai'):
    llm = get_llm(provider=provider)
    agent = create_react_agent(llm=llm, tools=tools, prompt=PROMPT)
    executor = AgentExecutor(
        agent=agent,
        tools=tools,
        callbacks=[handler],
        verbose=True,
        max_iterations=6,
        handle_parsing_errors=True,
    )
    return executor

agent = build_agent(provider='openai')  # Change provider here!
print('✅ Agent assembled and ready')

## Part 6: Human-in-the-Loop
The agent found some results — now a human decides what to do with them.

> **Key Insight:** Automation doesn't mean no humans.
> HITL means the RIGHT decisions go to humans, routine ones don't.

In [ ]:
def human_approval_gate(risk_summary, risk_level='MEDIUM'):
    """Pause and ask for human approval before alerting the team."""
    print('\n' + '='*55)
    print('🔔 ALERT REQUIRES YOUR APPROVAL')
    print('='*55)
    print(f'Risk Level: {risk_level}')
    print(f'\nSummary:\n{risk_summary[:500]}')
    print('\n' + '-'*55)
    
    response = input('\nSend alert to operations team? (yes/no): ').strip().lower()
    if response in ('yes', 'y'):
        print('✅ Alert approved!')
        return True
    else:
        print('🚫 Alert discarded.')
        return False

print('✅ Human-in-the-Loop gate ready')
print()
print('WORKSHOP EXERCISE:')
print('  Modify human_approval_gate() to:')
print('  1. Only ask for approval on HIGH risk (skip MEDIUM/LOW)')
print('  2. Log all decisions to a file with a timestamp')
print('  3. Add a third option: "escalate" that Cc\'s a manager')

## Part 7: Run the Full Agent
This cell runs the complete pipeline:
Search → Reason → Assess Risk → Human Approval → Alert

In [ ]:
# WORKSHOP: Try different queries and observe how the agent reasons
queries = [
    'Are there any port strikes or closures in Canada this month?',
    'Are there semiconductor shortages affecting electronics supply chains?',
    'What is the current status of the Panama Canal for shipping?',
]

query = queries[0]  # Change index to try different queries
print(f'Running query: {query}\n')

result = agent.invoke({'input': query})
output = result.get('output', '')

print('\n' + '='*55)
print('AGENT ASSESSMENT:')
print(output)

# Trigger HITL only for high-risk findings
if 'HIGH' in output.upper():
    human_approval_gate(risk_summary=output, risk_level='HIGH')
else:
    print('\n📊 No high-risk signals — logged automatically.')

## Part 8: Model Swap Challenge
The ultimate test — change the provider and re-run without touching anything else.

In [ ]:
# WORKSHOP CHALLENGE:
# 1. Run this cell with provider='openai' — note the response style
# 2. Change to provider='anthropic' — run again
# 3. Change to provider='google' — run again
# 4. Discuss: What was different? What was the same?

PROVIDER = 'openai'  # ← CHANGE THIS: 'openai' | 'anthropic' | 'google'

agent_v2 = build_agent(provider=PROVIDER)
result = agent_v2.invoke({
    'input': 'Search for any disruptions affecting automotive parts supply chains in North America.'
})
print(result.get('output', ''))

## Workshop Summary

You built an agent that demonstrates four enterprise-grade patterns:

| Pattern | Where in the code | Real-world benefit |
|---|---|---|
| **Human-in-the-Loop** | `human_approval_gate()` | Operations team reviews before alerts go out |
| **Model Flexibility** | `get_llm(provider=...)` | Switch vendors in seconds, not weeks |
| **No Vendor Lock-In** | Agent uses `BaseChatModel` interface | Works with any future LLM without rewriting |
| **Observability** | `WorkshopObservabilityHandler` | Full audit trail of every decision |

### Next Steps
1. Add a tool that reads from your internal supplier database
2. Connect the approval gate to Slack instead of console input
3. Enable LangSmith tracing and explore the visual dashboard
4. Add memory so the agent remembers past risk assessments